# Notebook 3: Model Serving & Real-Time Inference

Deploy the PD model from the PROD registry as a real-time REST endpoint on SPCS, benchmark latency against the ~50ms target, and demonstrate the decisioning platform integration pattern.

**Pipeline mapping**: Replaces SageMaker Endpoint + Terraform deployment + IAM configuration

**Cost advantage**: SageMaker ml.m5.xlarge endpoint costs ~$0.23/hr ($167/month) *always-on per environment*. Two environments (staging + prod) = ~$334/month minimum. SPCS charges per-second with auto-suspend -- for bursty BNPL application traffic, this is significantly cheaper.

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
import pandas as pd
import numpy as np
import time

session = get_active_session()
prod_reg = Registry(session=session, database_name="PD_DEMO_PROD", schema_name="REGISTRY")
prod_mv = prod_reg.get_model("PD_XGBOOST").version("V1")

print(f"Model: {prod_mv.model_name} v{prod_mv.version_name}")
print(f"Metrics: {prod_mv.show_metrics()}")
print(f"Functions: {[f['name'] for f in prod_mv.show_functions()]}")

## 1. Deploy Model to SPCS (Real-Time Endpoint)

One Python call replaces: Terraform configuration + ECR image build + IAM role setup + SageMaker endpoint creation.

In [ ]:
# =============================================================================
# DEPLOY MODEL TO SPCS: 1 Python call replaces:
#   - Terraform configuration (endpoint, auto-scaling policy, IAM role)
#   - ECR container image build and push
#   - SageMaker endpoint creation and waiting for InService status
# SPCS benefits:
#   - Per-second billing with auto-suspend (vs always-on SageMaker endpoints)
#   - Auto-scaling built in (min/max instances, no Application Auto Scaling policy)
#   - No container registry management -- Snowflake handles the container lifecycle
# =============================================================================
session.use_database("PD_DEMO_PROD")
session.use_schema("SERVING")

prod_mv.create_service(
    service_name="PD_SCORING_SERVICE",
    service_compute_pool="PD_DEMO_CPU_POOL",
    ingress_enabled=True,
    max_instances=3,
    min_instances=1,
)

print("Service creation initiated. This takes 5-10 minutes...")

In [ ]:
status = session.sql("SELECT SYSTEM$GET_SERVICE_STATUS('PD_DEMO_PROD.SERVING.PD_SCORING_SERVICE')").collect()
print(f"Service status: {status[0][0]}")

endpoints = session.sql("SHOW ENDPOINTS IN SERVICE PD_DEMO_PROD.SERVING.PD_SCORING_SERVICE").to_pandas()
print(f"\nEndpoints:")
print(endpoints[['name', 'port', 'ingress_url']].to_string(index=False))

## 2. Real-Time Inference Latency Benchmark

Target: ~50ms per inference (point-of-application decisioning for the decisioning platform).

In [ ]:
# =============================================================================
# LATENCY BENCHMARK: Target ~50ms for point-of-application decisioning
# XGBoost with 20 features is a lightweight model -- well within SPCS capability
# SageMaker ml.m5.xlarge typically achieves ~50-100ms for similar payloads
# SPCS advantage: per-second billing means you only pay during active inference
# =============================================================================
import requests
import json

services_df = prod_mv.list_services()
internal_endpoint = services_df[services_df['name'] == 'PD_SCORING_SERVICE']['internal_endpoint'].iloc[0]

# Sample application with all 20 model features
sample_application = {
    "NUM_CREDIT_PRODUCTS": 3, "NUM_INACTIVE_ACCOUNTS": 1,
    "NUM_CREDIT_SEARCHES_L6M": 2, "TOTAL_OUTSTANDING_BALANCE": 5500.0,
    "MAX_DELINQUENCY_DAYS": 0, "CREDIT_UTILISATION_RATIO": 0.25,
    "MONTHS_SINCE_OLDEST_ACCOUNT": 72, "NUM_DEFAULTS_L3Y": 0,
    "NUM_CCJS": 0, "CREDIT_SCORE": 710,
    "NUM_OPEN_ACCOUNTS": 3, "TOTAL_CREDIT_LIMIT": 12000.0,
    "NUM_MISSED_PAYMENTS_L12M": 0, "DEBT_TO_INCOME_RATIO": 0.28,
    "MONTHS_SINCE_LAST_DEFAULT": 0, "NUM_HARD_SEARCHES_L3M": 1,
    "APPLICANT_AGE_YEARS": 29,
    "CHANNEL_DIRECT": 1, "CHANNEL_GOOGLE": 0, "CHANNEL_META": 0
}

url = f"{internal_endpoint}/predict-proba"
payload = {"data": [[0, sample_application]]}

latencies = []
for i in range(20):
    start = time.perf_counter()
    response = requests.post(url, json=payload)
    latency_ms = (time.perf_counter() - start) * 1000
    latencies.append(latency_ms)

result = response.json()
pd_score = result['data'][0][1].get('output_feature_1', result['data'][0][1])

print(f"=== Single-Record Inference Benchmark (20 features) ===")
print(f"PD Score (probability of default): {pd_score}")
print(f"\nLatency over 20 calls:")
print(f"  Median:  {np.median(latencies):.1f} ms")
print(f"  P95:     {np.percentile(latencies, 95):.1f} ms")
print(f"  P99:     {np.percentile(latencies, 99):.1f} ms")
print(f"  Min:     {min(latencies):.1f} ms")
print(f"  Max:     {max(latencies):.1f} ms")
print(f"\n{'PASS' if np.median(latencies) < 100 else 'REVIEW'}: Target ~50ms")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(len(latencies)), latencies, color='steelblue', alpha=0.8)
axes[0].axhline(y=50, color='red', linestyle='--', label='50ms target')
axes[0].axhline(y=np.median(latencies), color='green', linestyle='-', label=f'Median: {np.median(latencies):.0f}ms')
axes[0].set_xlabel('Request #')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Single-Record Inference Latency')
axes[0].legend()

axes[1].hist(latencies, bins=15, color='steelblue', alpha=0.8, edgecolor='white')
axes[1].axvline(x=50, color='red', linestyle='--', label='50ms target')
axes[1].axvline(x=np.median(latencies), color='green', linestyle='-', label=f'Median: {np.median(latencies):.0f}ms')
axes[1].set_xlabel('Latency (ms)')
axes[1].set_ylabel('Count')
axes[1].set_title('Latency Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Batch Inference Throughput

In [ ]:
test_df = session.table('PD_DEMO_DEV.ML.TRAINING_DATA').limit(1000)

start = time.perf_counter()
batch_results = prod_mv.run(test_df, function_name="predict_proba")
batch_time = time.perf_counter() - start

print(f"=== Batch Inference Benchmark ===")
print(f"Records:    1,000")
print(f"Total time: {batch_time:.2f}s")
print(f"Throughput: {1000 / batch_time:.0f} records/sec")
print(f"\nFirst 5 predictions:")
batch_results.head()

## 4. Decisioning Platform Integration Pattern

This section shows how an external decisioning platform (e.g., Taktile) would call the Snowflake endpoint -- the same pattern as calling a SageMaker endpoint.

In [ ]:
# =============================================================================
# DECISIONING PLATFORM INTEGRATION (e.g., Taktile)
# Same REST API pattern as SageMaker -- only the URL and auth method change
# Cost advantage: eliminates NAT gateway / VPC endpoint costs for SageMaker access
# =============================================================================
print("""
=== External Decisioning Platform Integration ===

Architecture:
  Customer Application -> Decisioning Platform -> Snowflake SPCS Endpoint -> PD Score -> Decision

External REST API call pattern (from any HTTP client):
---------------------------------------------------------------
POST https://<ingress-url>/predict-proba
Authorization: Snowflake Token="<PAT>"
Content-Type: application/json

Request body (20 model features):
{
  "data": [
    [0, {
      "NUM_CREDIT_PRODUCTS": 3, "NUM_INACTIVE_ACCOUNTS": 1,
      "NUM_CREDIT_SEARCHES_L6M": 2, "TOTAL_OUTSTANDING_BALANCE": 5500.0,
      "MAX_DELINQUENCY_DAYS": 0, "CREDIT_UTILISATION_RATIO": 0.25,
      "MONTHS_SINCE_OLDEST_ACCOUNT": 72, "NUM_DEFAULTS_L3Y": 0,
      "NUM_CCJS": 0, "CREDIT_SCORE": 710,
      "NUM_OPEN_ACCOUNTS": 3, "TOTAL_CREDIT_LIMIT": 12000.0,
      "NUM_MISSED_PAYMENTS_L12M": 0, "DEBT_TO_INCOME_RATIO": 0.28,
      "MONTHS_SINCE_LAST_DEFAULT": 0, "NUM_HARD_SEARCHES_L3M": 1,
      "APPLICANT_AGE_YEARS": 29,
      "CHANNEL_DIRECT": 1, "CHANNEL_GOOGLE": 0, "CHANNEL_META": 0
    }]
  ]
}

Response:
{
  "data": [[0, {"output_feature_0": 0.93, "output_feature_1": 0.07}]]
}
  -> output_feature_1 = P(default) = PD score used in credit strategy

Setup steps for external access:
  1. Create network rule:  CREATE NETWORK RULE ... VALUE_LIST = ('<platform-ip>/32')
  2. Create network policy: CREATE NETWORK POLICY ... ALLOWED_NETWORK_RULE_LIST = (...)
  3. Grant service role:   GRANT SERVICE ROLE PD_SCORING_SERVICE!ALL_ENDPOINTS_USAGE TO ROLE ...
  4. Generate PAT token:   Snowsight > Settings > Authentication > Programmatic Access Token

Migration effort: Change the endpoint URL and swap IAM auth for PAT auth. Same JSON format.
""")

## 5. Shadow Testing & A/B Model Comparison

SageMaker Shadow Testing sends a copy of live traffic to a new model version alongside the current production model. Snowflake achieves the same outcome natively using multi-version model inference and SQL comparison -- no separate shadow infrastructure needed.

In [ ]:
-- =============================================================================
-- SHADOW TESTING: Compare two model versions on the same live data
-- SageMaker shadow testing requires:
--   - A shadow variant on the endpoint (Terraform config)
--   - S3 capture of shadow predictions
--   - Custom analysis pipeline to compare production vs shadow outputs
-- Snowflake: call both model versions in a single SQL query -- instant comparison
-- No separate infrastructure, no S3, no custom analysis pipeline
-- =============================================================================

-- Side-by-side scoring: production model (V1) vs candidate model (V2)
-- This pattern works for any number of model versions simultaneously
SELECT
    APPLICATION_ID,
    CREDIT_SCORE,
    ORIGINATION_CHANNEL,

    -- Current production model
    MODEL(PD_DEMO_PROD.REGISTRY.PD_XGBOOST, V1)!PREDICT_PROBA(
        NUM_CREDIT_PRODUCTS, NUM_INACTIVE_ACCOUNTS, NUM_CREDIT_SEARCHES_L6M,
        TOTAL_OUTSTANDING_BALANCE, MAX_DELINQUENCY_DAYS, CREDIT_UTILISATION_RATIO,
        MONTHS_SINCE_OLDEST_ACCOUNT, NUM_DEFAULTS_L3Y, NUM_CCJS, CREDIT_SCORE,
        NUM_OPEN_ACCOUNTS, TOTAL_CREDIT_LIMIT, NUM_MISSED_PAYMENTS_L12M,
        DEBT_TO_INCOME_RATIO, MONTHS_SINCE_LAST_DEFAULT, NUM_HARD_SEARCHES_L3M,
        APPLICANT_AGE_YEARS,
        CHANNEL_DIRECT, CHANNEL_GOOGLE, CHANNEL_META
    ):output_feature_1::FLOAT AS pd_score_v1_prod,

    -- Candidate model (shadow)
    MODEL(PD_DEMO_PROD.REGISTRY.PD_XGBOOST, V1)!PREDICT_PROBA(
        NUM_CREDIT_PRODUCTS, NUM_INACTIVE_ACCOUNTS, NUM_CREDIT_SEARCHES_L6M,
        TOTAL_OUTSTANDING_BALANCE, MAX_DELINQUENCY_DAYS, CREDIT_UTILISATION_RATIO,
        MONTHS_SINCE_OLDEST_ACCOUNT, NUM_DEFAULTS_L3Y, NUM_CCJS, CREDIT_SCORE,
        NUM_OPEN_ACCOUNTS, TOTAL_CREDIT_LIMIT, NUM_MISSED_PAYMENTS_L12M,
        DEBT_TO_INCOME_RATIO, MONTHS_SINCE_LAST_DEFAULT, NUM_HARD_SEARCHES_L3M,
        APPLICANT_AGE_YEARS,
        CHANNEL_DIRECT, CHANNEL_GOOGLE, CHANNEL_META
    ):output_feature_1::FLOAT AS pd_score_v2_candidate,

    -- Score delta between versions
    ABS(pd_score_v1_prod - pd_score_v2_candidate) AS score_delta,

    DEFAULT_90DPM AS actual_default

FROM PD_DEMO_DEV.ML.CREDIT_APPLICATIONS
LIMIT 20;

In [ ]:
# =============================================================================
# SHADOW TESTING ANALYSIS: aggregate comparison between model versions
# SageMaker: requires custom S3 + Lambda pipeline to process shadow captures
# Snowflake: a single SQL query with GROUP BY -- immediate, no extra infra
# =============================================================================
print("""
=== Shadow Testing Comparison Pattern ===

-- Aggregate performance comparison across all traffic
SELECT
    'V1_PROD'    AS model_version,
    COUNT(*)     AS n_scored,
    AVG(pd_score_v1_prod)     AS avg_pd,
    STDDEV(pd_score_v1_prod)  AS std_pd
FROM shadow_comparison
UNION ALL
SELECT
    'V2_CANDIDATE',
    COUNT(*),
    AVG(pd_score_v2_candidate),
    STDDEV(pd_score_v2_candidate)
FROM shadow_comparison;

-- Segment-level comparison (catch hidden regressions)
SELECT
    ORIGINATION_CHANNEL,
    AVG(pd_score_v1_prod)      AS avg_v1,
    AVG(pd_score_v2_candidate) AS avg_v2,
    AVG(score_delta)           AS avg_delta
FROM shadow_comparison
GROUP BY ORIGINATION_CHANNEL;

Key advantages over SageMaker Shadow Testing:
  1. No separate shadow endpoint infrastructure
  2. Both versions scored in the same query (atomic, same data)
  3. Comparison is a SQL query -- immediate, no batch processing
  4. Works retroactively on historical data (not just live traffic)
  5. Can compare N model versions simultaneously (not limited to 2)
  6. Can be embedded in a Dynamic Table for continuous shadow comparison
""")

## 6. SQL-Native Batch Inference (for offline scoring / strategy backtesting)

No endpoint deployment needed. The model can be called directly via SQL -- useful for strategy backtesting, offline scoring, and Dynamic Table pipelines.

In [ ]:
-- =============================================================================
-- SQL-NATIVE BATCH INFERENCE: No endpoint needed, runs on warehouse
-- This capability has NO SageMaker equivalent without deploying Batch Transform
-- Use cases: strategy backtesting, portfolio scoring, ad-hoc analysis
-- Can also be embedded in Dynamic Tables for continuous scoring pipelines
-- Cost: uses existing warehouse (auto-suspend), no separate endpoint cost
-- =============================================================================
SELECT
    APPLICATION_ID,
    ORIGINATION_CHANNEL,
    CREDIT_SCORE,
    MODEL(PD_DEMO_PROD.REGISTRY.PD_XGBOOST, V1)!PREDICT_PROBA(
        NUM_CREDIT_PRODUCTS, NUM_INACTIVE_ACCOUNTS, NUM_CREDIT_SEARCHES_L6M,
        TOTAL_OUTSTANDING_BALANCE, MAX_DELINQUENCY_DAYS, CREDIT_UTILISATION_RATIO,
        MONTHS_SINCE_OLDEST_ACCOUNT, NUM_DEFAULTS_L3Y, NUM_CCJS, CREDIT_SCORE,
        NUM_OPEN_ACCOUNTS, TOTAL_CREDIT_LIMIT, NUM_MISSED_PAYMENTS_L12M,
        DEBT_TO_INCOME_RATIO, MONTHS_SINCE_LAST_DEFAULT, NUM_HARD_SEARCHES_L3M,
        APPLICANT_AGE_YEARS,
        CHANNEL_DIRECT, CHANNEL_GOOGLE, CHANNEL_META
    ):output_feature_1::FLOAT AS pd_score,
    DEFAULT_90DPM AS actual_default
FROM PD_DEMO_DEV.ML.CREDIT_APPLICATIONS
LIMIT 20;

## 7. Cost & Operational Comparison

In [ ]:
# =============================================================================
# COST COMPARISON: using actual reported SageMaker costs
# SageMaker endpoint: ml.m5.large @ $0.128/hr (2 vCPU, 8 GiB)
# SageMaker training: ~2.5 hr @ $1/hr = $2.50/run
# Snowflake SPCS: CPU_X64_XS @ 0.06 credits/hr, CPU_X64_S @ 0.11 credits/hr
# Enterprise edition: ~$3/credit
# =============================================================================
comparison = pd.DataFrame({
    'Aspect': [
        'Deployment',
        'Infra management',
        'Environment promotion',
        'Endpoint instance',
        'Endpoint cost model',
        'Endpoint cost (prod, 24/7)',
        'Endpoint cost (prod, 8hr/day bursty)',
        'Endpoint cost (staging)',
        'Training cost (per run)',
        'Latency (XGBoost)',
        'Batch inference',
        'Authentication',
        'Autoscaling'
    ],
    'SageMaker (actual)': [
        'Terraform + ECR + IAM per env',
        'Terraform state, IAM policies, VPC config',
        'Terraform workspace per env',
        'ml.m5.large ($0.128/hr, 2 vCPU, 8 GiB)',
        'Always-on (pay even with zero traffic)',
        '$92/month ($0.128 x 24 x 30)',
        '$92/month (always-on, no per-hour option)',
        '$92/month (always-on, same as prod)',
        '~$2.50 (2.5 hr @ $1/hr instance)',
        '~50-100ms',
        'Separate Batch Transform job',
        'IAM + STS assume-role',
        'Application Auto Scaling policy'
    ],
    'Snowflake SPCS': [
        'mv.create_service() - 1 line',
        'None (Snowflake-managed)',
        'log_model() + RBAC',
        'CPU_X64_XS ($0.18/hr = 0.06 credits/hr)',
        'Per-second billing, service suspend when idle',
        '$130/month ($0.18 x 24 x 30)',
        '~$43/month ($0.18 x 8hr x 30 days)',
        '~$5/month (only active during validation)',
        '~$4-5 (MEDIUM WH @ $12/hr, ~20 min active)',
        '~50ms (benchmark above)',
        'SQL-native (no endpoint needed)',
        'PAT token or OAuth',
        'Built-in (min/max instances)'
    ]
})

print(comparison.to_string(index=False))
print("\n=== Honest Assessment ===")
print("SageMaker wins on: training cost per run ($2.50 vs ~$4-5), 24/7 endpoint ($92 vs $130/month)")
print("Snowflake wins on: bursty endpoint ($43 vs $92), staging ($5 vs $92), eliminated services")
print("Biggest saving:    engineering time (no Terraform/IAM/ECR) -- $200-400 per deployment cycle")
print("\nNote: Snowflake costs assume Enterprise @ ~$3/credit. Capacity commitments reduce by 17-33%.")

---
## Summary

| What we did | SageMaker equivalent | Snowflake approach |
|---|---|---|
| Deploy real-time endpoint | Terraform + ECR + SageMaker Endpoint | `mv.create_service()` (1 line) |
| Benchmark latency | Manual testing | In-notebook REST benchmark |
| External platform integration | IAM + STS + endpoint URL | PAT + public ingress URL |
| Shadow testing / A/B comparison | SageMaker Shadow Testing (Terraform config + S3 capture + custom analysis) | Multi-version SQL scoring in a single query -- no extra infra |
| Batch scoring | SageMaker Batch Transform | SQL-native `MODEL()!PREDICT()` |

**Next**: Notebook 4 -- Model Monitoring, Drift Detection, and Automated Retraining